In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from natsort import natsorted
import numpy as np
import re
import sys

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]




In [2]:
dir = rf"./data/diffraction_monitor_data"
os.makedirs(dir, exist_ok=True)

In [3]:
try:
    data_path = f"{dir}/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5"
    data_old = AM.read_hdf5_as_dict(data_path)
    print(data_old.keys())
except FileNotFoundError:
    print("File not found.")
    data_old = {}
except Exception as e:
    print(f"An error occurred: {e}")
    data_old = {}

folder_path = rf"../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction"

Error reading HDF5 file: [Errno 2] Unable to synchronously open file (unable to open file: name = './data/diffraction_monitor_data/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
dict_keys([])


In [4]:
reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "reference.txt"),get_ref=False)

amps_ref = reference_object.sim_data["diffraction"].amps
Pinc = np.abs(amps_ref.sel(orders_x=0, orders_y=0, polarization="p"))**2

reference_exit = reference_object.sim_data["flux1"].flux


Configured successfully.


16:33:07 W. Europe Daylight Time Billed flex credit cost: 0.109.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [5]:
n_values = list(data_old['n_values'] )if 'n_values' in data_old.keys() else []
ff_values =list( data_old['ff']) if 'ff' in data_old.keys() else []
size_values = list(data_old['sizes'])if 'sizes' in data_old.keys() else []
z_values = list(data_old['z_values']) if 'z_values' in data_old.keys() else []
values = data_old['transmission_data'] if 'transmission_data' in data_old.keys() else {}
reference_entry=None

# Polar/azimuthal angle of every diffraction order, dims (orders_x, orders_y, f).
# These depend only on the transverse period, the Bloch vector, the frequency list and
# the medium at the monitor -- none of which vary in this sweep (only the slab thickness
# changes, and that is along z). So capture them once and check the rest agree.
class DiffractionGeometryError(RuntimeError):
    """Raised when a sim's diffraction grid differs from the one the angle mask was built on."""

theta_orders = None
phi_orders = None
diffraction_sim_size = None
_angle_geometry = None

# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      try:
        z_value = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', dirpath).group(1))
      except AttributeError:
        print(f"Could not extract z_value from directory: {dirpath}")
        continue
      
      z_values.append(z_value)
      for filename in filenames:
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ffr_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            sample = float(re.search(r'sample_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            try:
                test_val = values[n_value][ff][z_value][size][sample]
                print(f"Data for n={n_value}, ff={ff}, z={z_value}, size={size}, sample={sample} already exists. Skipping file: {filename}")
            except KeyError:
                #Retrieve simulation data 
                if os.path.isfile(os.path.join(dirpath, filename)):
                  file=os.path.join(dirpath, filename)
                  structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
                  sim_data_i = structure_1.sim_data
                  transmission_entry = sim_data_i['flux2'].flux
                  transmission_exit = sim_data_i['flux1'].flux

                  #Diffraction-order angles: identical for every sim, so store the first set
                  diffraction_i = sim_data_i["diffraction"]
                  geometry_i = (tuple(np.ravel(diffraction_i.sim_size)),
                                tuple(np.ravel(diffraction_i.bloch_vecs)),
                                tuple(np.ravel(diffraction_i.f)[[0, -1]]),
                                len(diffraction_i.f))
                  if theta_orders is None:
                      theta_orders, phi_orders = diffraction_i.angles
                      diffraction_sim_size = tuple(np.ravel(diffraction_i.sim_size))
                      _angle_geometry = geometry_i
                      print(f"Diffraction geometry captured from {filename}: "
                            f"transverse period {diffraction_sim_size} um, "
                            f"{theta_orders.sizes['orders_x']} x {theta_orders.sizes['orders_y']} orders")
                  elif geometry_i != _angle_geometry:
                      raise DiffractionGeometryError(
                          f"Diffraction geometry differs from the stored one in {filename}: "
                          f"{geometry_i} vs {_angle_geometry}. The angle mask is not valid here.")
                 
                  if str(n_value) not in values.keys():
                    values[str(n_value)] = {}
                  if str(ff) not in values[str(n_value)].keys():
                      values[str(n_value)][str(ff)] = {}
                  if str(z_value) not in values[str(n_value)][str(ff)].keys():
                        values[str(n_value)][str(ff)][str(z_value)] = {}
                  if str(size) not in values[str(n_value)][str(ff)][str(z_value)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)] = {}
                  if str(sample) not in values[str(n_value)][str(ff)][str(z_value)][str(size)].keys():
                        values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)] = {}

                  #p = co and s = cross
                  amps = structure_1.sim_data["diffraction"].amps
                  lambdas = td.C_0 / structure_1.sim_data["diffraction"].f
                  T_total = transmission_exit/reference_exit

                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["amps"] = amps
                  values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]["T_total"] = T_total

        except DiffractionGeometryError:
            # must NOT be swallowed by the catch-all below: a wrong angle mask would
            # silently corrupt every aperture quantity downstream
            raise
        except Exception as e:
            print("Error:", e)
            continue

# Fall back to the empty reference if every file was already cached (loop body skipped)
if theta_orders is None:
    diffraction_ref = reference_object.sim_data["diffraction"]
    theta_orders, phi_orders = diffraction_ref.angles
    diffraction_sim_size = tuple(np.ravel(diffraction_ref.sim_size))
    print(f"No new sims loaded; diffraction angles taken from the reference "
          f"(transverse period {diffraction_sim_size} um)")

# Define unconditionally: cell 8 stores `lambdas`, but the in-loop assignment above only
# happens when at least one file is actually read.
lambdas = td.C_0 / np.asarray(theta_orders.f)
       






Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction
Could not extract z_value from directory: ../../../data/20260708 LSU Transmission n_3.4 ff_0.237 ffh_0.185 Schulz Diffraction\n_3.40
Configured successfully.


16:33:13 W. Europe Daylight Time Billed flex credit cost: 3.735.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Diffraction geometry captured from LSU_ffr_0.2369_size_0.8741258741258742_n_3.40_z_100.0_sample_0.txt: transverse period (11.44, 11.44) um, 11 x 11 orders
Configured successfully.


16:33:19 W. Europe Daylight Time Billed flex credit cost: 3.735.

...

16:45:08 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

Configured successfully.


16:45:14 W. Europe Daylight Time Billed flex credit cost: 3.667.

                                 Note: the task cost pro-rated due to early     
                                 shutoff was below the minimum threshold, due to
                                 fast shutoff. Decreasing the simulation        
                                 'run_time' should decrease the estimated, and  
                                 correspondingly the billed cost of such tasks.

In [6]:
# Collection aperture at normal incidence: numerical aperture NA, i.e. sin(theta) <= NA.
NA_APERTURE = 0.25

if theta_orders is None:
    raise RuntimeError("theta_orders is not defined -- run the loading cell above first.")

# xarray aligns with an INNER join, so a mismatched coordinate silently drops data instead
# of raising. Check the grids agree exactly before relying on that.
def _same_grid(a, b, dims=("f",)):
    return all(np.array_equal(np.asarray(a.coords[d]), np.asarray(b.coords[d])) for d in dims)

if not _same_grid(theta_orders, Pinc):
    raise ValueError("Pinc is on a different frequency grid than the diffraction angles.")

# Evanescent orders carry theta = NaN, which compares False, so they drop out on their own.
# NB: tidy3d's DataArray overrides __eq__ to return a plain bool -- never use `==` for
# elementwise tests here; `<=`, `>`, `&` are the normal xarray ones.
# This cone always contains the specular (0,0) order (theta = 0), so it is never empty.
aperture = np.sin(theta_orders) <= NA_APERTURE

Lx_period, Ly_period = diffraction_sim_size
lambda_orders = td.C_0 / theta_orders.f
n_orders_aperture = aperture.sum(dim=("orders_x", "orders_y"))

# --- Why the aperture is NOT a plain sum of the orders inside the cone --------------------
# Each order stands for a CELL of directions of solid angle lambda^2/(Lx Ly cos theta), not
# for a pencil. Summing whole cells is a quadrature that is only right when the cells tile
# the cone; here one cell can be several times larger than the whole cone (0.489 sr at
# lambda=8 um), so a plain sum overstates the collected power at long lambda and understates
# it at short lambda. Checked against a periodic random-phase-screen model sampled 16x more
# finely in angle: plain sum +171% at lambda=8 um, the treatment below 0%.
#
# So each component is handled according to what it physically is:
#   * COHERENT light (ballistic + any coherent off-axis order) is a true delta function in
#     angle for a periodic structure, so summing it over the cone is exact -- no rescaling.
#   * DIFFUSE light is a continuum. Estimate its radiance from the channels and integrate
#     over the true cone. Per-order power = radiance * lambda^2/(Lx Ly) with the cos(theta)
#     cancelling exactly, so every order gives the same radiance estimate, and the cone
#     integral of a constant radiance is int_cone cos(theta) dOmega = pi NA^2.
omega_cone_nominal = np.pi * NA_APERTURE**2          # = int_cone cos(theta) dOmega
cell_solid_angle = lambda_orders**2 / (Lx_period * Ly_period)
omega_aperture = ((lambda_orders**2 / (Lx_period * Ly_period) / np.cos(theta_orders))
                  .where(aperture).sum(dim=("orders_x", "orders_y")))

print(f"NA = {NA_APERTURE} (theta_max = {np.rad2deg(np.arcsin(NA_APERTURE)):.2f} deg): "
      f"orders in cone min {int(n_orders_aperture.min())}, max {int(n_orders_aperture.max())}; "
      f"{100 * float((np.asarray(n_orders_aperture) < 2).mean()):.1f}% of frequencies hold the "
      f"specular order only (first ring enters at lambda <= {NA_APERTURE * Lx_period:.3f} um).")
print(f"  cells cover {float(omega_aperture.min()):.3f}-{float(omega_aperture.max()):.3f} sr vs "
      f"pi*NA^2 = {omega_cone_nominal:.3f} sr -> a plain channel sum would be off by "
      f"{100 * (float(omega_aperture.min()) / omega_cone_nominal - 1):+.0f}% to "
      f"{100 * (float(omega_aperture.max()) / omega_cone_nominal - 1):+.0f}%")

values_processed = {}
for n in values.keys():
    print(f"Processing n={n}")
    for ff in values[n].keys():
        for z_value in values[n][ff].keys():
            for size in values[n][ff][z_value].keys():
                samples   = list(values[n][ff][z_value][size].keys())
                amps_list = [values[n][ff][z_value][size][s]["amps"] for s in samples]
                T_total_average = np.mean([values[n][ff][z_value][size][s]["T_total"] for s in samples], axis=0)
                N         = len(amps_list)

                for a in amps_list:
                    if not _same_grid(a, theta_orders, dims=("f", "orders_x", "orders_y")):
                        raise ValueError(f"order/frequency grid mismatch at "
                                         f"n={n} ff={ff} z={z_value} size={size}")

                # coherent amplitude average (the ballistic / delta-function part)
                amps_average = sum(amps_list) / N

                # incoherent intensity averages <|a|^2>, all orders
                T_co    = sum(abs(a.sel(polarization="p"))**2 for a in amps_list) / N / Pinc
                T_cross = sum(abs(a.sel(polarization="s"))**2 for a in amps_list) / N / Pinc
                T_co_total    = T_co.sum(dim=("orders_x", "orders_y"))
                T_cross_total = T_cross.sum(dim=("orders_x", "orders_y"))

                T_ballistic = np.abs(
                    amps_average.sel(orders_x=0, orders_y=0, polarization="p"))**2/ Pinc

                # --- coherent power collected by the cone: delta functions, summed exactly --
                T_co_aperture_coh    = ((abs(amps_average.sel(polarization="p"))**2 / Pinc)
                                        .where(aperture).sum(dim=("orders_x", "orders_y")))
                T_cross_aperture_coh = ((abs(amps_average.sel(polarization="s"))**2 / Pinc)
                                        .where(aperture).sum(dim=("orders_x", "orders_y")))

                # --- diffuse: Bessel-corrected sample variance per order -------------------
                # (1/(N-1)) sum |a_i - abar|^2 is the UNBIASED diffuse power. Writing it as
                # <|a|^2> - |<a>|^2 would be low by (1 - 1/N) = 0.8 at N = 5.
                if N < 2:
                    print(f"  WARNING: only {N} realization(s) at n={n} ff={ff} z={z_value} "
                          f"size={size} -- diffuse part undefined, set to NaN")
                    T_co_aperture_diff    = T_co_aperture_coh * np.nan
                    T_cross_aperture_diff = T_cross_aperture_coh * np.nan
                else:
                    var = sum(abs(a - amps_average)**2 for a in amps_list) / (N - 1)
                    # every order in the cone gives the same radiance estimate (cos theta
                    # cancels), so average over them to cut the 1/sqrt(N-1) noise
                    I0_co    = ((var.sel(polarization="p") / Pinc).where(aperture)
                                .mean(dim=("orders_x", "orders_y")) / cell_solid_angle)
                    I0_cross = ((var.sel(polarization="s") / Pinc).where(aperture)
                                .mean(dim=("orders_x", "orders_y")) / cell_solid_angle)
                    T_co_aperture_diff    = I0_co * omega_cone_nominal
                    T_cross_aperture_diff = I0_cross * omega_cone_nominal

                T_co_aperture    = T_co_aperture_coh + T_co_aperture_diff
                T_cross_aperture = T_cross_aperture_coh + T_cross_aperture_diff

                # naive channel sum, kept only as a cross-check where the cone holds many orders
                T_co_aperture_chansum    = T_co.where(aperture).sum(dim=("orders_x", "orders_y"))
                T_cross_aperture_chansum = T_cross.where(aperture).sum(dim=("orders_x", "orders_y"))

                if str(n) not in values_processed.keys():
                    values_processed[str(n)] = {}
                if str(ff) not in values_processed[str(n)].keys():
                    values_processed[str(n)][str(ff)] = {}
                if str(z_value) not in values_processed[str(n)][str(ff)].keys():
                      values_processed[str(n)][str(ff)][str(z_value)] = {}
                if str(size) not in values_processed[str(n)][str(ff)][str(z_value)].keys():
                      values_processed[str(n)][str(ff)][str(z_value)][str(size)] = {}

                entry = values_processed[str(n)][str(ff)][str(z_value)][str(size)]
                entry["T_ballistic"] = T_ballistic
                entry["T_co"] = T_co_total
                entry["T_cross"] = T_cross_total
                entry["T_total"] = T_total_average
                entry["T_co_aperture"] = T_co_aperture
                entry["T_cross_aperture"] = T_cross_aperture
                entry["T_co_aperture_coh"] = T_co_aperture_coh
                entry["T_cross_aperture_coh"] = T_cross_aperture_coh
                entry["T_co_aperture_diff"] = T_co_aperture_diff
                entry["T_cross_aperture_diff"] = T_cross_aperture_diff
                entry["T_co_aperture_chansum"] = T_co_aperture_chansum
                entry["T_cross_aperture_chansum"] = T_cross_aperture_chansum


NA = 0.25 (theta_max = 14.48 deg): orders in cone min 1, max 9; 60.0% of frequencies hold the specular order only (first ring enters at lambda <= 2.860 um).
  cells cover 0.063-0.489 sr vs pi*NA^2 = 0.196 sr -> a plain channel sum would be off by -68% to +149%
Processing n=3.4


In [7]:
# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)
z_unique = np.unique(z_values)

In [8]:
values_processed['3.4']['0.2369'].keys()

dict_keys(['100.0', '5.0'])

In [9]:

data = {
    "transmission_data": values_processed,
    "n_values": n_unique,
    "ff_values": ff_unique,
    "size_values": size_unique,
    "z_values": z_unique,
    "lambdas": lambdas,
    # aperture bookkeeping, indexed like the other arrays (ascending frequency)
    "NA_aperture": np.array([NA_APERTURE]),
    "n_orders_aperture": n_orders_aperture,       # orders inside the cone per frequency
    "omega_aperture": omega_aperture,             # solid angle those cells actually cover
    "omega_cone_nominal": np.array([omega_cone_nominal]),   # pi*NA^2 = int_cone cos(th) dOmega
}

In [11]:
create_hdf5_from_dict(data,data_path)

In [12]:
data_path

'./data/diffraction_monitor_data/20260708_average_diffraction_n_3.4_ff_0.237_ffh_0.185_schulz.h5'